In [2]:
from agents import Agent, Runner
from agents.decorators import function_tool
from ingest import load_search_index
import json

In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [4]:
@function_tool
def search(query: str) -> str:
    """
    Search the FAQ database for entries matching the given query.
    """
    index = load_search_index()

    filter_dict = {"course": "llm-zoomcamp"}
    boost_dict = {"question": 2.0, "section": 0.5}

    results = index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
        )

    return json.dumps(results, indent=2)

In [5]:
agent = Agent(
    name="Course Assistant",
    model="gpt-5.4",
    instructions=instructions,
    tools=[search],
)

In [6]:
async def run_agent(question: str):
    result = await Runner.run(
        agent,
        question
    )
    return result.final_output

In [1]:
question = "What is the deadline for the final project submission?"

In [9]:
print(await run_agent(question))

The final project (capstone) deadline is shown on the current cohort’s course management platform, next to the submission form:

https://courses.datatalks.club/llm-zoomcamp-2026/

The exact dates vary by cohort and by attempt, so please check there rather than using dates from past cohorts.

Are there other areas that you want to explore?
